

Group nr: 42

Name 1 and CID: Alfred Sändare Svensson (sandare)

Name 2 and CID: Felix Holm Pileborg (pileborg)

In [1]:
import numpy as np
import copy
import random
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from mining_world import Environment
from IPython.display import Image

from numpy.typing import NDArray
Matrix = NDArray[np.float64]
groupNumber = 42


pygame 2.6.1 (SDL 2.28.4, Python 3.12.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


# Introduction

This assignment dels with data structures, K-NN, tree-based classifiers and evaluation of the methods deployed on a fictional scenario.  

## Mining world 

<img src="imgs/poster.png" width="800"/>

## Scenario


Humanity has now reached a point where we need to extract and refine more Copium, a precious resource with great value. The only problem is that Copium can only be found on certain uninhabitable planets. This of course means that automated robots are sent instead.      

Copium is naturally very unstable and is only exists very temporary before it decays. There are very specific geological activities and circumstances needed for copium to form. The life cycle of Copium follows. First, a hot stream of liquid magma flows to the surface, creating a hotspot that that looks like a small crater. At the surface, if the conditions are correct, copium can form during the cool-down period. But as stated previously, Copium is unstable in its natural environment and decays to other materials shortly after. 

The formation of these deposits craters are very random, but the heat from them can easilly be detected with a satellite. But there is no way of knowing if the newly formed depoist contains copium from just a satellite, therefor there is a robot rover on the ground with sensors that can collect further measurements. The rover has many sensors that can measure the properties of the ground below it, but of course, Copium can not directly be detected with these types of sensors. This is where the machine learning approach will be used, to take all those measurements and try to classify if the deposit contains Copium or not.  


<img src="imgs/overView.png" width="500"/>


# Navigation - Tree search

This section will show how the naivigation is done. This is not a part of the assignment to understand, but will be used. 

##  Breadth first

The method used is a breadth first search algorithm, it is one of the simplest tree search algorithms and basically tries every option for a fixed number of steps and chooses the best one. The robot only have 4 actions, it can move at every time step in a chosen direction. 

## Actions 

<img src="imgs/actions.png" width="300"/>


In [2]:
class Node():
    def __init__(self, actor):
        self.actor = actor
        self.total_score = 0
    
    def update(self, action, inherited_score):
        score = self.actor.step(action)
        self.total_score = 1.05*inherited_score + score
        return self.total_score
    
    def get_score(self):
        return self.total_score

In [3]:
def breadth_first_search(actor, max_depth, action_space):
    node = Node(copy.deepcopy(actor)) 
    queue_keys = ['0'] # queue to keep track of nodes that has not yet been expanded.  
    visited = {queue_keys[0]: node} # saves visited nodes in order to not recalulate the entire path for each step. 
    
    max_score = -np.inf
    best_action = None

    while True:
        key = queue_keys.pop(0)
        if len(key) > max_depth: # stop at a set depth 
            break    
        node = visited[key]
        
        for action in action_space: # expand all children nodes
            child_node = copy.deepcopy(node)  # copy current node
            score = child_node.update(action=action, inherited_score=node.get_score()) # update node with action
            child_key = key + action # create child node key
            
            if score > max_score: # save best path 
                max_score = score
                best_action = child_key[1]
                
            visited[child_key] = child_node  # add child node to visited nodes.
            queue_keys.append(child_key)  # add child node queue of non expanded nodes. 
            
    return best_action


# Collect data

The first step is to collect data that will be used for both training and validation. The available types features can be seen with `env.get_sensor_properties()` and the actual measurements can be retrieved with `env.get_sensor_readings()`. It will return a dictionary with the same keys as in `env.get_sensor_properties()` containg a value for each feature. If the robot is not currently over a deposit, then it will return `None`. The true label can be extracted with `env.get_ground_truth()`, which will return a 1 if there is copium in the deposit and 0 if not. 
  

In [4]:
# Have True for first time running and then set to False. 
collect_data = False

In [5]:
env = Environment(map_type=1, fps=500, resolution=(1000, 1000))
env.exit()
sensor_properties = env.get_sensor_properties()
print('Sensor properties', sensor_properties)


Sensor properties ['ground_density', 'moist', 'reflectivity', 'silicon_rate', 'oxygen_rate', 'iron_rate', 'aluminium_rate', 'magnesium_rate', 'undetectable']


In [6]:
if collect_data == True:
    env = Environment(map_type=1, fps=500, resolution=(1000, 1000))
    sensor_properties = env.get_sensor_properties()

    # We can initilize the dictionary the following way.
    data = dict()
    data['copium'] = [] 
    for key in sensor_properties:
        data[key] = []

    for i in range(5000):
        action = breadth_first_search(actor=env.get_actor(), max_depth=3, action_space=env.get_action_space())
        env.step(action)
        # if we are over a deposit. 
        if env.get_sensor_readings() is not None:
            sensor_readings = env.get_sensor_readings()
            copium = env.get_ground_truth()

            for key in sensor_readings:
                data[key].append(sensor_readings[key])
            data['copium'].append(copium)

        env.render()

    env.exit()


# Exerscie 1: Data structure

## a) Pandas data frame 
In this assignment we will work pandas data frame for storing the collected data. First create a pandas data frame from the dictionary. The documentation for it can be found at https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html, only the data feild needs to be filled in with the created dictionary. Call this data frame df. 


In [7]:
# TODO: create pandas dataframe

if collect_data == True:
    df = pd.DataFrame(data=data)
    print(df)


## b) Save data
Collecting data for each time the notebook is opened is time consuming and the tests are not as reproducable if the data is different each time. Therefore we save the dateframe and we can load the data instead. After this you can set collect_data=False above. Use the function pandas function df.to_csv('name_of_file.csv', index=False)

The task is to save the data in a file named mydata.csv

In [8]:

if collect_data == True:
    # TODO: save data to a csv file named mydata.
    df.to_csv("mydata.csv", index=False)

## c) Load data
The task is to load the data from mydata.csv into a dataframe named df. Use the function `df = pd.read_csv('name_of_file.csv')`

In [9]:

# TODO: read data from mydata.csv into a dataframe called df.  
df = pd.read_csv("mydata.csv")

In [10]:
# From the data frame you can access all data for a key with for example:
print("All data for a feature \n", df["copium"])
print()

# You can access a single sample with:
print("Single sample from index \n", df.iloc[1])
print()

# You can access all freatures but one with:
all_features_without_copium = df.drop(columns='copium')
print("All features without copium \n", all_features_without_copium)

# To get an overview 
df.describe()

All data for a feature 
 0       0
1       0
2       0
3       0
4       0
       ..
1552    0
1553    0
1554    0
1555    0
1556    0
Name: copium, Length: 1557, dtype: int64

Single sample from index 
 copium            0.000000
ground_density    0.565280
moist             0.163534
reflectivity      0.186647
silicon_rate      0.117167
oxygen_rate       0.016440
iron_rate         0.618648
aluminium_rate    0.169801
magnesium_rate    0.047720
undetectable      0.030224
Name: 1, dtype: float64

All features without copium 
       ground_density     moist  reflectivity  silicon_rate  oxygen_rate  \
0           1.923067  0.191268      0.042675      0.202305     0.038098   
1           0.565280  0.163534      0.186647      0.117167     0.016440   
2           2.231129  0.176790      0.379341      0.103445     0.024423   
3           1.136273  0.232682      0.532672      0.038844     0.017649   
4           1.791873  0.012970      0.188444      0.178837     0.018053   
...              ... 

,copium,ground_density,moist,reflectivity,silicon_rate,oxygen_rate,iron_rate,aluminium_rate,magnesium_rate,undetectable
count,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000,1557.000000
mean,0.165703,1.362617,0.149779,0.303366,0.136924,0.040864,0.408607,0.253512,0.110746,0.049347
std,0.371934,0.603689,0.086138,0.174574,0.077105,0.027844,0.156549,0.127912,0.066202,0.029348
min,0.000000,0.400000,0.000025,0.000028,0.013041,0.000060,0.047266,0.022130,0.005378,0.005215
25%,0.000000,0.920114,0.076813,0.147385,0.076801,0.018387,0.291362,0.150426,0.060159,0.026543
50%,0.000000,1.274357,0.151020,0.294149,0.131233,0.037893,0.431496,0.249178,0.106245,0.045714
75%,0.000000,1.698775,0.222553,0.457267,0.182866,0.057275,0.524714,0.343470,0.149664,0.066004
max,1.000000,4.084972,0.299957,0.599525,0.564162,0.201902,0.826430,0.766683,0.488686,0.210330


## d) Part 1: Split data

Here we will devide the data into a training set and a test set. Good rule of thumb is to use 80% of the data in the training set and 20 % in the test set. The the two data sets should be randomly sampled (shuffle=True), we also set he seed with random_state so that the split is reproducable. This is done with train_test_split() from sklearn, https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html. The syntax looks like 

train, test = train_test_split(dataframe, test_size=ratio_test_set, shuffle=True, random_state=42)

Why is it important that the data is shuffled when it is split, what could happen otherwise?

**Text answer:**

In case the data is dependent on time, the datasets could differ, if not shuffled.




In [11]:
# devide the data into train and test set. 
trainData, testData = train_test_split(df, test_size=0.2, shuffle=True, random_state=groupNumber)


## e) Part 1: Analyse data balance

The occurance can be retrevied with .value_counts() from a pandas data frame. Here get the occurance of copium in the samples. Is the dataset balanced?

In [12]:
# TODO: Print number of samples with copium and the number of samples without copium.
occurance = float( df["copium"].value_counts()[1] / len(df["copium"]) )

print("occurance: ", occurance)


occurance:  0.16570327552986513


**Text answer:**

The dataset is not balanced. Most data points do not contain copium :(


## e) Part 2: Balance data

We will throughout this assignment test each method with the original dataset and a balanced dataset. We show how balancing can be done by downsampling below. Your task is to create an upsampled balanced dataset. You only need to use the upsampled data set for the rest of the other part 2) exerices. 

What could be the reason for chosing either upsampled or downsampeled method to balance the dataset?


**Text answer:**

Upsampling may use more storage space and make the model slower, while downsampling discards information.

In [13]:
# TODO balance data det. 
# Seperate the data into something that contains copium and one that doesn't,
# can for example be done with train[train["copium"]==0] etc.
train_zero = trainData[trainData["copium"]==0]
train_one = trainData[trainData["copium"]==1]

# downsample majority
train_zero_downsampled = resample(
    train_zero,
    n_samples=train_one.shape[0]
)
train_balanced_downsampled = pd.concat([train_one, train_zero_downsampled])


# upsample minority 
train_one_upsampled = resample(
    train_one,
    n_samples=train_zero.shape[0]
)
train_balanced_upsampled = pd.concat([train_one_upsampled, train_zero])


# Exercise 2: Performance evaluation

Here we will define a class that later will be used for evaluation the performance of the classification methods. It will be used to log the accuracy, precsision and recall. More information about precision and recall can be found at https://en.wikipedia.org/wiki/Precision_and_recall. 

Explain why the different metics are usefull and why is not always accuarcy enough?

**Text answer:**

In some cases it may be useful to know which type of misclassification is probable. 



In [14]:
class Classification_eval(object):
    def __init__(self):
        # counters 
        self.TP = 0 # Predicted positive, positive label
        self.FP = 0 # Predicted positive, negative label
        self.TN = 0 # Predicted negative, negative label
        self.FN = 0 # Predicted negative, positive label. 
    
    def update(self, pred, label):
        """
        pred - is the prediction will be either a 1 or 0. 
        label - is the correct answer, will be either a 1 or 0.
        """
        # add to one of the correct counter each time this function is called. 
        if (pred==1):
            if (pred==label): self.TP +=1
            else: self.FP +=1
        else:
            if (pred==label): self.TN +=1
            else: self.FN +=1
        

    def accuracy(self): 
        # returns the accuracy 
        if (self.TP + self.TN) == 0:
            return 0
        
        accuracy = (self.TP + self.TN)/(self.TP + self.TN + self.FP + self.FN)
        return np.round(accuracy, 4)
    
    def precision(self): # percentage of the estimated positive that actually is positive
        if (self.TP + self.FP) == 0:
            return 0
        
        precision = self.TP / (self.TP + self.FP)
        return np.round(precision, 4)
    
    def recall(self): # percentage of correctly identified positive of the total positive
        if (self.TP + self.FN) == 0:
            return 0

        recall = self.TP / (self.TP + self.FN)
        return np.round(recall, 4)

In [15]:
# Test Classification_eval

test_eval = Classification_eval()
test_eval.update(1,1)
test_eval.update(1,1)
test_eval.update(1,1)
test_eval.update(0,1)
test_eval.update(1,0)
test_eval.update(1,0)
test_eval.update(0,0)
test_eval.update(0,0)
test_eval_ = np.allclose([test_eval.accuracy(), test_eval.precision(), test_eval.recall()], [0.625, 0.6, 0.75])
if test_eval_:
    print('Test successful')
else:
    print('Test failed')
assert test_eval_
   


Test successful


# Exercise 3: K- nearest neighbours

## a) Normalize

Here we will code our K-NN classifier, method 2.1 on page 21 in the book has the psudo code for K-NN. We will start with the data normalization, i.e. we will normalize the input data so that each feature has the same range in terms of max/min values. The min value can be found with data.min(), similarly for the max value. 

Why is it important that the data is normalized for the K-NN algorithm?

**Text answer:**

Becaue we want the contribution of all dimensions of the input data to be weighed equally when computing distance


In [16]:
class Normalize(object):
    def __init__(self):
        self.min = None
        self.max = None
    
    def normalize(self, data):
        # normalize the data and return it. 
        return (data-self.min)/(self.max-self.min)
    
    def update_normalization(self, data):
        # Save the min and max values for each feature. This funciton is only used for the training data.
        self.min = data.min()
        self.max = data.max()
        return self.normalize(data)

    

## b) K-NN
Lets make the k-Nearest Neighbors (k-NN) algorithm, fill in the TODO. 

The core idea behind k-NN is to make predictions based on the "neighborhood" of data points in a feature space. In this impementation we will use the $l^2$-norm for distance to determine the k nearest samples. The classification is done with a majoiry vote of the k nearest samples.

In [17]:


class KNN(object):
    def __init__(self, k):
        self.features: pd.DataFrame = None # normalized features from training data 
        self.labels: pd.DataFrame = None # the corresponding labels (if there is copium)
        self.normalize = Normalize() # class instance for normalization
        self.k = k # the k value in k-nn algorithm.
        
    def fit(self, features, labels):
        # This is where we save the training data. 
        # TODO: update the normalize filter and normalize the features (save to self.features)
        # and save the labels to self.labels. Hint see cell bellow for how the inputs looks like
        
        self.features = self.normalize.update_normalization(features)
        self.labels = labels

    
    def predict(self, sample):
        # For this part you are not allowed to use an existing K-NN package. 
        # This function takes one sample as an input and outputs a predicion 
        
        # TODO: 
        # First normalize the input sample. 
        # Find the k closest samples in self.features to the input sample
        # Return the prediction from the majority vote.  
        x_star: Matrix = self.normalize.normalize(sample).to_numpy()
        X: Matrix = self.features.to_numpy()
        Y: Matrix = self.labels.to_numpy()
    
        distances: list[float] = [ np.linalg.norm(X[i, :] - x_star).item() for i in range(X.shape[0])]

        nearest_indices = np.argsort(distances)
        Y_kNN: Matrix = np.array([ Y[i] for i in nearest_indices[0:self.k] ])


        # if more than half of k-nearest contain copium, return true
        return ((sum([self.labels.values[nearest_indices[i]] for i in range(self.k)]) / self.k)) > 1/2


In [18]:
# Test the KNN against known values. The test only covers small sample and there are hidden tests which will 
# tested after submitting. 

test_train = pd.read_csv('test.csv')
test_test = pd.read_csv('test2.csv')
train_labels = test_train['copium']
train_features = test_train.drop(columns='copium')
test_labels = test_test['copium']
test_features = test_test.drop(columns='copium')

knn = KNN(k=5)
knn.fit(train_features, train_labels)
pred_list = []

for i in range(test_features.shape[0]):
    pred = knn.predict(test_features.iloc[i])
    pred_list.append(pred)
val = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
if np.allclose(pred_list, val):
    print('Test successful')
else:
    print('Test failed')
    

    


Test successful


## c) part 1: Evaluate the K-NN 
Evaluate the K-NN and choose a suitable k value. 

In [19]:
#train_labels = trainData['copium']
#train_features = trainData.drop(columns='copium')

train_labels = train_balanced_upsampled['copium']
train_features = train_balanced_upsampled.drop(columns='copium')

y = testData['copium']
x = testData.drop(columns='copium')

# TODO, try differenent values of k. 
knn = KNN(k=50)
knn.fit(train_features, train_labels)

log = Classification_eval()
for i in range(x.shape[0]):
    pred = knn.predict(x.iloc[i])
    log.update(pred, y.iloc[i])

print('Accuarcy', log.accuracy())
print('Precision', log.precision())
print('Recall', log.recall())


Accuarcy 0.8718
Precision 0.537
Recall 0.6591


Try some differnet values of k and just looking at these resutlts would the klassifier work well for all k? 

Answer:

| k | Accuracy | Precision | Recall | 
| --- | --- | --- | --- |
| 1  | 0.8718 | 0.5714 | 0.3636 |  
| 5  | 0.8782 | 0.875  | 0.1591 |
| 20 | 0.8654 | 1      | 0.0455 |   
| 50 | 0.859  | 0      | 0      |  


## c) part 2, balanced data. 
Now try with the upsampled balanced data, test for the same k values as in part 1. Are the results different and whitch one would identify more copium?  

Answer:


| k | Accuracy | Precision | Recall | 
| --- | --- | --- | --- |
| 1  | 0.8718 | 0.5714 | 0.3636 |
| 5  | 0.8494 | 0.4717 | 0.5682 | 
| 20 | 0.8494 | 0.4737 | 0.6136 |   
| 50 | 0.8494 | 0.4737 | 0.6136 |

## d) kNN with Sklearn
Typically one would not implement these algorithms by hand for each time you need them. Here we show how the kNN algorithm from sklearn can be used. Note, it does not normalize the features by default. The advantanges with the sklearn version is in computational cost as they utilize clever tree structures to minimize the search problem. 

A quick note: Even if ready packages exist, it is still very valuable to understand the principels behind the algorithms. For example when the learning doesn't work as well as you liked (happens more often then not...), you can reason to why and modify the problem or method into something that does work.  

In [20]:
# Set up data used for fitting the model
train_labels = trainData['copium']
train_features = trainData.drop(columns='copium')

# with the off the shelf model we need normalize the data before 
normalize = Normalize()
data_features = normalize.update_normalization(train_features)

# The test data
y = testData['copium']
x = testData.drop(columns='copium')

# Normalize the test data
x_norm = normalize.normalize(x)

# Set up an knn with sklearn 
knn_sklearn = KNeighborsClassifier(n_neighbors=5)
knn_sklearn.fit(data_features.values, train_labels.values)

# Set up our own for comparison 
knn = KNN(k=5)
knn.fit(train_features, train_labels)

# Test both for the 20 first values
for i in range(20):
    pred = knn.predict(x.iloc[i])
    pred_sklearn = knn_sklearn.predict([x_norm.iloc[i]])
    print('Prediciton from our model', pred, ' and from sklearn ',pred_sklearn)


Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model True  and from sklearn  [1]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model False  and from sklearn  [0]
Prediciton from our model True  and from sklearn  [1]
Prediciton f

# Exercise 4: Tree based classifier

In this exercise we will implement a tree based classifier. The implementation will be a class representing a node and will recursivly create child node until the entire tree structure is built. The general step for learning a tree structure from learning data is as follows:

1. Initiate the the root node.
2. Call the learn function with the learning data. 

Inside the learn function

3. Check if all the data has the same classification, and check if the data set is smaller then min_node_size. If any of these are true, set the node as a leaf node and return. 
4. Find the spliting point in the data that returns the smallest gini value, be careful to not use the label which we want to classify. 
5. Split the dataset, one is usually called head and the other tail. Save the splitting parameter and which value was used for the splitting. 
6. Check the prediction of the splited data sets (majority copium or not) and create two child nodes, one for each of the splited dataset. 
7. Call the the learn function of the child node with the corresponding split data and appdend the child node. After that return. 

The above will build the tree structure. To use the tree for prediction simply call the predict function of the root node with a data sample. In the predict function the following steps should happen.

1. Check if the current node is a lead node, if it is simply return the self.prediction of that node. 
2. If it is not a leaf node then check the value of the splitting parameter in the data sample and compare with the splitting value. 
3. Call the predict function of the correspoinding child node with the data sample and return the answer.

This will recurivly traverse the tree until a leaf node containing the prediction is reached. 

 


## 4 a) Find split point

First we define a function that can find the splitting point with the lowest gini value for a feature. For example, what is the best way to split the data with regards to the reflectivity feature. Page 34 in the book describes Gini index.  

The gini value can be described as:

If $\Gamma$ contains the set of all labels, then $\Gamma(x < c)$ would be all labels that belong to the criteria $x < c$, where $x$ is a feature or parameter and $c$ is the splitting criteia (split_value).  


For example: 

If $x$ is `iron_rate` and our splitting critera is $c = 0.4$, then $\Gamma(x < c)$ would be a set of all the label from samples with $\textrm{iron\_rate} < 0.4$. Remeber that our labels are either 0 or 1 in this assignment. 


Then we can define:

$r_1 = mean(\Gamma(x < c))$

$r_2 = mean(\Gamma(x \geq c))$

Gini index left side

$q_1 = 2r_1(1-r_1)$

Gini index right side

$q_2 = 2r_2(1-r_2)$

We define $len(\Gamma)$ to give the number of samples in the dataset, then the weighted gini value is:

$q = \frac{len(\Gamma(x_i < c))}{len(\Gamma)}*q_1 + \frac{len(\Gamma(x_i \geq c))}{len(\Gamma)}*q_2$

The goal is to find a criteria $c$ that minimizes the gini value $q$ with regards to a feature. One way to do it is to sort the data with regards to the feature, this makes $\Gamma(x < c)$ easier. We could then try every split point with a for loop and save the one that has the lowes gini value. The data is always split beteen two data points therefore the critieria c would be the average between the two data points that are sorted, i.e. $c_i = \frac{x_{i-1} + x_i}{2}$. First split has index 1. 

In [21]:

def find_split_point(data: pd.DataFrame, label: str, feature: str) -> \
    tuple[float, float, pd.DataFrame, pd.DataFrame]:
    """
    Arguments:
        data - all the data we want to split, our (gamma)
        label - the parameter we want to classify. 
        feature - the feature we want to check for, our x_i, could for example be iron
    
    Returns:
        split_value - the spliting value, our c. 
        gini_value - the gini value for the best c.
        df_head - the data frame belonging to x_i < c
        df_tail - the data frame belonging to x_i => c
    """
    # Beging by sorting the data after the paramter. 
    sorted_data = data.sort_values(by=feature)
    sorted_label = sorted_data[label]
    
    
    # TODO loop through all the split points in the sorted data and find 
    # the best gini_value (s) and split_value (c).
    # the functions .head(split_index) and .tail(len(x) - split_index) could be useful. 

    q = float("inf")    # best found gini/cost/whatever
    index_ = 0          # where it was found
    for i in range(1, len(sorted_data[feature])):   # iterate over index of feature to split *before* (skip first)
        L = sorted_label.head(i)
        R = sorted_label.tail(len(sorted_data)-i)

        r1 = np.mean(L)
        r2 = np.mean(R)
        q1 = 2*r1*(1-r1)
        q2 = 2*r2*(1-r2)
        gini = (len(L)/len(sorted_label))*q1 + (len(R)/len(sorted_label))*q2
        if gini < q:
            q = gini
            index_ = i
        
    
    gini_value = q
    split_value = (sorted_data[feature].iloc[index_-1]+sorted_data[feature].iloc[index_])/2 # average of features next to split index
    
    # TODO: get the two data frames the corresponds to the split data. 
    # the functions .head(split_index) and .tail(len(x) - split_index) could be useful. 
    df_head = sorted_data.head(index_)
    df_tail = sorted_data.tail(len(sorted_data)-index_)
    
    return split_value, gini_value, df_head, df_tail

In [22]:
# Test for find_split_point function, the test does not test everything and more hidden tests are used 
# during grading. 

df_test = pd.read_csv('test.csv')
split_value, gini_value, df_head, df_tail = find_split_point(df_test, 'copium', 'ground_density')

if np.isclose(split_value, 1.8762144694722296, atol=1e-8) and np.isclose(gini_value, 0.27655122655122655, atol=1e-8):
    print('Test successful')
else:
    print('Test failed')
    print("split_value = ", split_value)
    print("gini_value = ", gini_value)
    
    

Test successful


## 4 b) Tree Node

Here we will code a tree node class that can recursivly call it self. 


In [23]:
class TreeNode():
    def __init__(self, prediction=None):
        self.split_value: float = None  # the splitting value (c)
        self.split_feature: str = None  # what feature where uesd for the split (x_i)
        self.child_nodes = []           # list that contains two child nodes, if not leaf_node
        self.prediction: bool | None = prediction # classification made in this node.

    def isLeaf(self) -> bool: return len(self.child_nodes) == 0
    def predict(self, data: pd.Series) -> bool:
        # TODO: To make a prediction we need to traverse the tree recursivly down to a leaf node and return 
        # the prediction. 
        # step 1: check if this is a leaf node, if it is then return prediction of this node, 
        # otherwise contine with step 2.
        if (self.isLeaf()): 
            assert self.prediction is not None
            return self.prediction

        # step 2: Compare the input data with the splitting criteria, i.e. the is the value of the data smaller 
        # or larger then the split value.
        # step 3: Call the prediction function in the corresponding child_node and return the prediction. 
        if (data[self.split_feature] < self.split_value):
            return self.child_nodes[0].predict(data)
        else:
            return self.child_nodes[1].predict(data)

        
            
    def learn(self, data: pd.DataFrame, label: str, min_node_size: int) -> None:        
        """
        data - the training data
        label - the parameter we want to classify, i.e. "copium"
        min_node_size - number of data points in a node for it to become a leaf node (exclusive lower bound fur some reason)
        """
        labels = data[label] 
        features = data.drop(columns=label)

        # TODO: wirte the learning function. 
        # Step 1: check if this node should be turned into a leaf node, this happens if the data set is smaller or equal 
        # then the min_node_size or if the data labels are homogenious, i.e. avg = 0 or 1.
        if len(labels) <= min_node_size or ( labels.mean() in [0, 1] ): return
        
        # Step 2: Loop over all features and get the best gini and split_value. Make sure you are not 
        # using the label as a feature. "data.columns" can give you all features plus the label 
        # Save the best split value and split_paramter and the two new data frames 
        # corresponding to the split [df_head, df_tail] (these will be data frames for the child nodes).

        lowestGini: float = float("inf")
        head: pd.DataFrame | None = None
        tail: pd.DataFrame | None = None
        for featureName in features.columns:
            # try a feature
            testSplit, testGini, testHead, testTail = find_split_point(data, label, featureName)
            # keep if better
            if (testGini < lowestGini):
                lowestGini = testGini
                self.split_feature = featureName
                self.split_value = testSplit
                head = testHead
                tail = testTail

        assert head is not None
        assert tail is not None
                
        # Step 3: Calculate the majority vote prediction of the two data frames.(hint: df_head[label].mean() > 0.5, could be useful) 
        # Create two child nodes, (e.g. child = TreeNode(prediction=1) for prediction copium) and and call the 
        # learn function with the corresponding dataframe.


        child1 = TreeNode(prediction = head[label].mean() > 0.5)
        child1.learn(head, label, min_node_size)

        child2 = TreeNode(prediction = tail[label].mean() > 0.5)
        child2.learn(tail, label, min_node_size)
        
        # Step 4: append the the child node to the self.child_nodes. It should be in the order 
        # of the child node correspoinding to [df_head, df_tail].

        self.child_nodes = [child1, child2]
        return

In [24]:
# Test to give indicaion if the TreeNode class is correct.There might be small variations in the implementation 
# that fails this part, but if it is only one thing in the prediction that is off, then it is probably still fine.  

df_test = pd.read_csv('test.csv')
tree = TreeNode() # create root node
# learn the tree structure
tree.learn(df_test, "copium", min_node_size=4)

pred_list = []
pred_answer = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 
               1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 
               0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0]

for i in range(df_test.shape[0]):
    pred = tree.predict(df_test.iloc[i])
    pred_list.append(pred)
    
if np.allclose(pred_list, pred_answer):
    print('Test successful')
else:
    def format(labels: list[bool]) -> list[str]:
        return [ str(label)+" " if label else str(label) for label in labels]
    print('Test failed')
    print('Prediction', format(pred_list))
    print('Correct   ', format([ bool(label) for label in pred_answer ]))



Test successful


##  Train the Tree Classifier

In [25]:
tree = TreeNode() # create root node

#tree.learn(trainData, "copium", min_node_size=50)
tree.learn(train_balanced_upsampled, "copium", min_node_size=50)


## Test Classifier

In [26]:
y = testData['copium']
x = testData.drop(columns='copium')
log = Classification_eval()

for i in range(x.shape[0]):
    pred = tree.predict(x.iloc[i])
    log.update(pred, y.iloc[i])
        
print('accuarcy', log.accuracy())
print('precision', log.precision())
print('recall', log.recall())

accuarcy 0.8301
precision 0.4444
recall 0.8182


## 4 c) part 1

Use the non balanced data to learn the tree classifier and test the classifier for some different `min_node_size` values.

Answer:



| min_node_size | Accuracy | Precision | Recall | 
| --- | --- | --- | --- |
| 2  | 0.9071 | 0.6531 | 0.7273 |  
| 10 | 0.9038 | 0.6458 | 0.7045 |   
| 20 | 0.9135 | 0.6809 | 0.7273 |   
| 50 | 0.8974 | 0.6034 | 0.7955 |  

## 4 c) part 2, Try with balanced data

Use the balanced data to learn the tree classifier and test the classifier for some different `min_node_size` values.

Answer:


| min_node_size | Accuracy | Precision | Recall | 
| --- | --- | --- | --- |
| 2  | 0.9167 | 0.6875 | 0.7500 |  
| 10 | 0.9231 | 0.7174 | 0.7500 |   
| 20 | 0.9038 | 0.6296 | 0.7727 |   
| 50 | 0.8590 | 0.5000 | 0.7955 |

## d) Tree classifier with sklearn 

Here we compare our tree classifier to the one implemented in sklearn. It is important to note that there can be differences between them, there can be minor implementation details that can affect the result, for example, there is some randomness in the sklearn implementaion. But overall they give simlar results. 

In [27]:
# Set up data used for fitting the model
train_labels = trainData['copium']
train_features = trainData.drop(columns='copium')

# Decition tree or classifier from sklearn 
tree_sklearn = DecisionTreeClassifier(min_samples_split=2, splitter='best', 
                             criterion='gini', min_samples_leaf=1)
# Train the sklearn tree classifer
tree_sklearn.fit(train_features.values, train_labels.values)

# Train our tree classifier 
tree = TreeNode() # create root node
tree.learn(trainData, "copium", min_node_size=2)

# Test both and compare
y = trainData['copium']
x = trainData.drop(columns='copium')
# classify the data
for i in range(30):
    pred_sklearn = tree_sklearn.predict([x.iloc[i]])
    pred = tree.predict(x.iloc[i])

    print('sklearn pred ',pred_sklearn, 'our tree pred ', pred)

sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [1] our tree pred  True
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [0] our tree pred  False
sklearn pred  [0] our tree pred  False
sklearn pred  [1] our tree pred  True
sklearn pred  [1] our tree pred  T

## Exersice 5: Balance data analysis

1. Is there a difference between the results of the balanced and non-balanced data for the tree classifier?
2. Is there a difference between the K-NN vs tree classifier?
3. Which one would you expect to be a faster classifier?

**Answer:**

1. There is no major difference.
1. k-NN has lower recall.
1. We'd expect the tree classifier to be faster, because navigating down a single tree branch is faster than computing distances.



## Exersice 6: Deployment (Optional)

Here we will try the learned classifiers on a larger map. Make sure that the last run version of K-NN and tree have good parameters i.e. k and min_node_size values. Which one found more copium?


In [29]:
env = Environment(map_type=2, fps=60, resolution=(1000, 1000))

try:
    sensor_properties = env.get_sensor_properties()
    sensor_sample = dict()
    for key in sensor_properties:
        sensor_sample[key] = [0]

    log_knn = Classification_eval()
    log_tree = Classification_eval()


    for i in range(500):
        action = breadth_first_search(actor=env.get_actor(), max_depth=3, action_space=env.get_action_space())
        env.step(action)
        if env.get_sensor_readings() is not None:
            sensor_readings = env.get_sensor_readings()
            for key in sensor_readings:
                sensor_sample[key][0] = sensor_readings[key]
            sensor_sample_df = pd.DataFrame(sensor_sample)
            log_knn.update(knn.predict(sensor_sample_df.iloc[0]), env.get_ground_truth())
            log_tree.update(tree.predict(sensor_sample_df.iloc[0]), env.get_ground_truth())
            env.plt_acc.update_acc(log_tree.accuracy(), log_knn.accuracy())
        env.render()

    env.exit()

    print("K-NN accuracy ", log_knn.accuracy(), "Tree accuracy", log_tree.accuracy())
    print("Number of copium deposits found, K-NN:", log_knn.TP, " Tree:", log_tree.TP)
except Exception as exc:
    env.exit()
    raise exc




K-NN accuracy  0.8396 Tree accuracy 0.8774
Number of copium deposits found, K-NN: 4  Tree: 13
